# Tutorial 05 — Flexible fiber dynamics

Reproduce two canonical flexible-fiber experiments from
**Delmotte, Climent & Plouraboué 2015** (*J. Comput. Phys.* **286**, 14–37):

1. **Figure 8(a)** — a straight fiber tumbling in pure shear flow. The fiber
   passes through a symmetric "S-shape" each half-period.
2. **Figure 10** — a fiber settling under gravity perpendicular to its long
   axis. After a transient "W" shape it converges to the steady
   "horseshoe".

Both motions are planar, so we use `FlexibleFiber(planar=True)`. The fiber
bends in the xz-plane and we use a custom shear flow `u = (γ̇·z, 0, 0)` so
that the shear plane and the bending plane coincide.

In [1]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import FlexibleFiber, Flow, FlowBodyRollout, gravity_field, no_flow

np.set_printoptions(precision=3, suppress=True)

## Helper — lab-frame bead positions

`FlowBodyRollout.rollout` returns the **body-reference** position and
orientation plus the per-bead DOFs. To plot the fiber shape we convert each
bead's body-frame position into the lab frame using the body Rodrigues vector.

In [2]:
def rodrigues_matrix(rvec):
    rvec = np.asarray(rvec, dtype=float)
    theta = np.linalg.norm(rvec)
    if theta < 1e-9:
        return np.eye(3)
    k = rvec / theta
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)


def lab_bead_positions(fiber, positions, orientations, dofs, design=None):
    """Return an array of shape (n_steps, n_beads, 3) with lab-frame bead positions."""
    if design is None:
        design = np.asarray(fiber.design_defaults)
    n_steps = int(positions.shape[0])
    n_beads = fiber.Nspheres
    t_dummy = np.array([0.0])
    out = np.zeros((n_steps, n_beads, 3))
    body_frame_positions = np.zeros((n_beads, 3))
    for step in range(n_steps):
        dof_step = np.asarray(dofs[step])
        for i in range(n_beads):
            body_frame_positions[i] = np.asarray(fiber.spheres[i].position(dof_step, design, t_dummy))
        R = rodrigues_matrix(orientations[step])
        out[step] = np.asarray(positions[step]) + body_frame_positions @ R.T
    return out

## Part 1 — Flexible fiber in shear flow (Figure 8a)

### Bending ratio

The paper parameterises fiber flexibility by the **bending ratio**
(Eq. 54): 

$$\mathrm{BR} = \frac{E (\ln 2 r_e - 1.5)}{\mu \dot\gamma^2 r_p^4}$$

with $r_p = N$ the aspect ratio (number of beads), $r_e = 0.7\,r_p$ the
equivalent ellipsoidal aspect ratio, and $E$ Young's modulus. SoftMobility
uses the **bending modulus** $K_b = E I$ directly, where $I = \pi a^4 / 4$
is the cross-section's second moment for a cylindrical fiber. Hence

$$K_b = \frac{\pi a^4}{4} \cdot \frac{\mathrm{BR}\,\mu\,\dot\gamma^2\,r_p^4}{\ln 2 r_e - 1.5}.$$

We keep the natural units $a = 1$, $\mu = 1$, $\dot\gamma = 1$. With
$\mathrm{BR} = 0.04$ and $N = 10$, this gives $K_b \approx 220$.

In [3]:
N = 10                # number of beads (kept small for tutorial responsiveness)
a = 1.0               # bead radius
mu = 1.0              # viscosity
shear_rate = 1.0      # shear rate γ̇
BR = 0.04             # paper Fig 8 bending ratio

rp = N
re = 0.7 * rp
K_b_shear = (np.pi * a**4 / 4) * BR * mu * shear_rate**2 * rp**4 / (np.log(2 * re) - 1.5)
print(f"K_b for BR=0.04, N={N}: {K_b_shear:.1f}")

K_b for BR=0.04, N=10: 275.8


### Build the fiber and the shear flow

We use a custom flow `u = (z, 0, 0)` (gradient in z, vorticity around y) so
the planar fiber bends in the same plane as the shear gradient. Mass is
zero — we ignore gravity for this experiment, but `gravity_field(g=0)`
still has to be supplied because `FlexibleFiber` registers a `gravity`
input.

In [4]:
fiber = FlexibleFiber(n_beads=N, radius=a, bending_rigidity=K_b_shear, mass=0.0, planar=True)
shear_xz = Flow(lambda pos, t: jnp.array([pos[2], 0.0, 0.0]))
rollout = FlowBodyRollout(
    soft_body=fiber,
    flow=shear_xz,
    input_map={"gravity": gravity_field(g=0.0)},
)
print(repr(fiber))

SPHERE ASSEMBLY
  10 spheres
  10 degrees of freedom
  4 design parameters
  3 input parameters

Default values
  degrees of freedom dof: ['theta_0', 'theta_1', 'theta_2', 'theta_3', 'theta_4', 'theta_5', 'theta_6', 'theta_7', 'theta_8', 'theta_9'] = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  design parameters param: ['radius', 'gap', 'K_b', 'mass'] = [  1.      0.05  275.806   0.   ]
  input parameters param: ['gravity0', 'gravity1', 'gravity2']

SPHERE 0
  radius: 1.0
  position: [0. 0. 0.]
  orientation: [0. 0. 0.]
  C_H:
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
  C_K:
[[   0.       0.       0.       0.       0.       0.       0.       0.
     0.       0.   ]
 [   0.       0.       0.       0.       0.       0.       0.       0.
     0.       0.   ]
 [   0.       0.       0.       0.       0.       0.       0.       0.
     0.       0.   ]
 [   0.       0.       0.       0.       0.       0.       0.       0.
     0.       0.   ]
 [-131.336  131.336    0.     

### Run the rollout

The fiber starts tilted by 30° in the xz-plane and straight (all bending
DOFs zero). The shear-flow rotation drives a Jeffery-like tumbling, and as
the fiber sweeps through the compressive quadrant it buckles into the
S-shape. We integrate over a few tumble periods.

The first call triggers JAX/XLA compilation (~10–20 s); subsequent calls
are sub-second.

In [5]:
dt_shear = 0.02
n_steps_shear = 3000   # final time = 60 = ~10 Jeffery periods (T = 2π(re+1/re)/γ̇ ≈ 44 for re=7)
init_orientation = jnp.array([0.0, np.pi / 6, 0.0])  # 30° tilt around ê_y

positions_s, orientations_s, dofs_s = rollout.rollout(
    dt=dt_shear,
    n_steps=n_steps_shear,
    init_orientation=init_orientation,
)
lab_pos_s = lab_bead_positions(fiber, positions_s, orientations_s, dofs_s)
L_fiber = (N - 1) * 2 * a * (1 + 0.05)
print(f"shape (n_steps, n_beads, 3): {lab_pos_s.shape}, L = {L_fiber:.2f}")

shape (n_steps, n_beads, 3): (3000, 10, 3), L = 18.90


### Plot — S-shape during a tumble (Figure 8a)

We follow the paper convention: shape snapshots in the xz-plane, in the
frame moving with the centre of mass, with axes scaled by the fiber length
$L$. Time labels follow the paper: a window of one half-tumble around the
compressive sweep.

In [6]:
# Pick snapshots that span one S-shape half-cycle.
step_indices = np.linspace(0, n_steps_shear - 1, 6, dtype=int)
times = step_indices * dt_shear

fig = go.Figure()
colors = ["#222222", "#444444", "#666666", "#888888", "#aaaaaa", "#cccccc"]
for color, step, t in zip(colors, step_indices, times, strict=False):
    snap = lab_pos_s[step]
    com = snap.mean(axis=0)
    centred = (snap - com) / L_fiber
    fig.add_trace(
        go.Scatter(
            x=centred[:, 0],
            y=centred[:, 2],
            mode="lines+markers",
            name=f"t = {t:.1f}",
            line=dict(color=color, width=2),
            marker=dict(size=6, color=color),
        )
    )

fig.update_layout(
    title=f"Fiber in shear flow (BR={BR}, N={N}) — reproduction of Fig 8a",
    xaxis_title="x / L",
    yaxis_title="z / L",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=700,
    height=600,
    plot_bgcolor="white",
)
fig.show()

## Part 2 — Settling fiber (Figure 10)

### Elasto-gravitational number

A fiber settling under transverse gravity is parameterised by the
**elasto-gravitational number** (paper Eq. 55):

$$B = \frac{F_\perp\,L}{K_b}, \qquad F_\perp = N\,m\,g.$$

Large $B$ means a floppy fiber. Figure 10 corresponds to $B = 10\,000$
(very floppy). Our **linear** discrete biharmonic does not bound the
per-joint angle, so once the bending exceeds ~$\pi/3$ adjacent beads start
to wrap around and the simulation diverges before the steady horseshoe is
reached. We therefore use $B = 1\,000$ — moderately floppy, large enough
to develop the metastable W shape and a U-like curl, but inside the
stability range of the linear model. Set $m = 1$, $g = 1$, $N = 10$, and
solve for $K_b$.

In [7]:
N_set = 10
m = 1.0
g = 1.0
B_target = 1_000.0

L_set = (N_set - 1) * 2 * a * (1 + 0.05)
F_perp = N_set * m * g
K_b_settle = F_perp * L_set / B_target
print(f"K_b for B={B_target:g}, N={N_set}: {K_b_settle:.4f}")

K_b for B=1000, N=10: 0.1890


In [8]:
fiber_set = FlexibleFiber(
    n_beads=N_set,
    radius=a,
    bending_rigidity=K_b_settle,
    mass=m,
    planar=True,
)
rollout_set = FlowBodyRollout(
    soft_body=fiber_set,
    flow=no_flow(),
    input_map={"gravity": gravity_field(g=g)},
)
print(repr(fiber_set))

SPHERE ASSEMBLY
  10 spheres
  10 degrees of freedom
  4 design parameters
  3 input parameters

Default values
  degrees of freedom dof: ['theta_0', 'theta_1', 'theta_2', 'theta_3', 'theta_4', 'theta_5', 'theta_6', 'theta_7', 'theta_8', 'theta_9'] = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  design parameters param: ['radius', 'gap', 'K_b', 'mass'] = [1.    0.05  0.189 1.   ]
  input parameters param: ['gravity0', 'gravity1', 'gravity2']

SPHERE 0
  radius: 1.0
  position: [0. 0. 0.]
  orientation: [0. 0. 0.]
  C_H:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
  C_K:
[[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]
 [-0.09  0.09  0.    0.    0.    0.    0.    0.    0.    0.  ]
 [ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.  ]]

SPHERE 1
  radius: 1.0
  p

### Run the rollout

The fiber is initially horizontal (along $\hat e_x$) and gravity pulls it
in $-\hat e_z$. The terminal settling speed is empirically $V_s\!\sim\!0.13$
in our units (single-sphere Stokes overestimates the drag once
hydrodynamic interactions between beads are included), so the
characteristic time is $\tau = L/V_s \approx 150$. We integrate over a few
$\tau$ to see the W shape develop and the U-curl progress.

In [9]:
dt_set = 0.5
n_steps_set = 1500   # final time = 750 ≈ 5 τ

positions_g, orientations_g, dofs_g = rollout_set.rollout(
    dt=dt_set,
    n_steps=n_steps_set,
)
lab_pos_g = lab_bead_positions(fiber_set, positions_g, orientations_g, dofs_g)
print("max |dof| over trajectory:", float(np.max(np.abs(dofs_g))))
print("final centre of mass:", lab_pos_g[-1].mean(axis=0))

max |dof| over trajectory: 1.5748860836029053
final centre of mass: [   9.446    0.    -118.922]


### Plot — W shape and late curl (Figure 10)

Two snapshots, in the frame moving with the centre of mass, scaled by the
fiber length $L$:

* an early time showing the metastable **W shape** (paper Fig. 10a);
* a late time showing the **U-curl** that precedes the steady horseshoe
  (full horseshoe convergence at $B = 10\,000$ is unreachable with our
  linear bending — see notes at the end).

In [10]:
# Pick snapshot indices: early (W) and late (U-curl).
step_W = int(0.20 * n_steps_set)   # W forms early
step_U = n_steps_set - 1            # late: U-curl

snap_W = lab_pos_g[step_W]
snap_U = lab_pos_g[step_U]
centred_W = (snap_W - snap_W.mean(axis=0)) / L_set
centred_U = (snap_U - snap_U.mean(axis=0)) / L_set

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f"\"W\" shape  (t ≈ {step_W*dt_set:.0f})",
                    f"U-curl  (t ≈ {step_U*dt_set:.0f})"),
    horizontal_spacing=0.12,
)
for col, snap in [(1, centred_W), (2, centred_U)]:
    fig.add_trace(
        go.Scatter(
            x=snap[:, 0], y=snap[:, 2],
            mode="lines+markers",
            line=dict(color="black", width=3),
            marker=dict(size=8, color="black"),
            showlegend=False,
        ),
        row=1, col=col,
    )
    fig.update_xaxes(title_text="(x − x_c) / L", row=1, col=col, range=[-0.6, 0.6])
    fig.update_yaxes(
        title_text="(z − z_c) / L",
        row=1, col=col,
        scaleanchor=f"x{'' if col == 1 else col}",
        scaleratio=1,
        range=[-0.6, 0.6],
    )

fig.update_layout(
    title=f"Settling fiber (B={B_target:g}, N={N_set}) — qualitative match to Fig 10",
    width=900,
    height=500,
    plot_bgcolor="white",
)
fig.show()

## Notes

* `FlexibleFiber` uses **rigid bonds** (paper Joint Model, Eqs. 2–4)
  parameterised by per-bead orientation DOFs, plus a **linear discrete
  biharmonic** bending torque. This captures the qualitative shapes of
  Figs. 8 and 10 but does **not** bound the per-joint angle, so
  large-deformation steady states (the full $B = 10\,000$ horseshoe of
  Fig. 10b, the buckled rod of Fig. 8b with $\kappa^{eq} \neq 0$) require
  the geometric curvature formula (paper Eq. 33) to stay stable — out of
  scope for this tutorial.
* For closer agreement with the paper, increase `N` to ~20–30 (the
  asymptotic regime, paper Fig. 9a). Construction time scales nearly
  linearly; rollout JIT compile is the dominant cost.
* The custom xz-shear flow is needed because `softmobility.shear_flow`
  defaults to gradient in y, while `FlexibleFiber(planar=True)` bends in
  xz.